# Notebook 2 — Data Preprocessing & Outlier Flagging

Notebook này sử dụng trực tiếp dữ liệu từ `MolaDatabase.invoices.json` và `MolaDatabase.invoice_items.json`, sau đó tạo bộ dữ liệu sạch ở mức hóa đơn để phục vụ EDA và mô hình hóa.

Mục tiêu:
- làm sạch giá trị thiếu
- chuẩn hóa kiểu dữ liệu
- tạo biến đặc trưng
- gắn cờ outlier theo IQR
- xuất file `prepared_invoice_dataset.csv` và `prepared_invoice_dataset_cleaned.csv`

In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()


def resolve_path(*candidates):
    for name in candidates:
        p = BASE_DIR / name
        if p.exists():
            return p
    raise FileNotFoundError(f"Không tìm thấy file nào trong danh sách: {candidates}")


def load_json_dataframe(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.json_normalize(data)


def categorize_service(item_name: str) -> str:
    text = str(item_name).lower()
    if 'kiểm định' in text:
        return 'Inspection'
    if 'giám định' in text:
        return 'Survey'
    if 'hiệu chuẩn' in text:
        return 'Calibration'
    if 'thử nghiệm' in text or 'phân tích' in text:
        return 'Testing'
    return 'Other'


In [ ]:
# Nạp lại dữ liệu gốc và dựng dataset mức hóa đơn
invoices_path = resolve_path('MolaDatabase.invoices.json')
items_path = resolve_path('MolaDatabase.invoice_items.json')

df_invoices = load_json_dataframe(invoices_path).rename(columns={
    'unique_key': 'invoice_unique_key',
    'issue_date': 'invoice_date',
    'buyer.name': 'buyer_name',
    'buyer.tax_code': 'buyer_tax_code',
    'financial_summary.subtotal_before_tax': 'subtotal_before_tax',
    'financial_summary.total_tax': 'total_tax',
    'financial_summary.total_amount': 'total_amount',
    'processing_info.status': 'invoice_status',
    'processing_info.payment_method': 'payment_method',
}).copy()

df_items = load_json_dataframe(items_path).rename(columns={'subtotal': 'item_subtotal'}).copy()

df_invoices['invoice_date'] = pd.to_datetime(df_invoices['invoice_date'], errors='coerce')
for col in ['subtotal_before_tax', 'total_tax', 'total_amount', 'item_count']:
    df_invoices[col] = pd.to_numeric(df_invoices[col], errors='coerce')
for col in ['quantity', 'unit_price', 'item_subtotal', 'tax_amount', 'discount']:
    df_items[col] = pd.to_numeric(df_items[col], errors='coerce')

df_items['service_category'] = df_items['item_name'].apply(categorize_service)
product_items = df_items[df_items['item_type'].eq('product_service')].copy()

joined = product_items.merge(
    df_invoices[['invoice_unique_key', 'invoice_number', 'invoice_date', 'buyer_name', 'buyer_tax_code',
                 'subtotal_before_tax', 'total_tax', 'total_amount', 'payment_method', 'invoice_status', 'item_count']],
    on='invoice_unique_key', how='left'
)

df = (
    joined.groupby('invoice_unique_key', as_index=False)
    .agg(
        invoice_number=('invoice_number', 'first'),
        invoice_date=('invoice_date', 'first'),
        buyer_name=('buyer_name', 'first'),
        buyer_tax_code=('buyer_tax_code', 'first'),
        subtotal_before_tax=('subtotal_before_tax', 'first'),
        total_tax=('total_tax', 'first'),
        total_amount=('total_amount', 'first'),
        payment_method=('payment_method', 'first'),
        invoice_status=('invoice_status', 'first'),
        item_count=('item_count', 'first'),
        total_quantity=('quantity', 'sum'),
        avg_unit_price=('unit_price', 'mean'),
        max_item_subtotal=('item_subtotal', 'max'),
        service_category=('service_category', lambda s: s.mode().iat[0] if not s.mode().empty else 'Other')
    )
)

print('Dataset hóa đơn trước làm sạch:', df.shape)
display(df.head())

In [ ]:
# Xử lý giá trị thiếu và tạo đặc trưng thời gian
for col in ['buyer_name', 'buyer_tax_code', 'payment_method', 'invoice_status', 'service_category']:
    df[col] = df[col].fillna('Unknown')

for col in ['subtotal_before_tax', 'total_tax', 'total_amount', 'item_count', 'total_quantity', 'avg_unit_price', 'max_item_subtotal']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

df['month'] = df['invoice_date'].dt.to_period('M').astype(str)
df['quarter'] = df['invoice_date'].dt.quarter.fillna(0).astype(int)
df['year'] = df['invoice_date'].dt.year.fillna(0).astype(int)
df['is_cancelled'] = df['invoice_status'].str.contains('hủy', case=False, na=False).astype(int)
df['payment_method_group'] = np.where(
    df['payment_method'].str.contains('chuyển khoản|ck', case=False, na=False),
    'Bank Transfer',
    np.where(df['payment_method'].str.contains('tiền mặt|tm', case=False, na=False), 'Cash', 'Other')
)
df['log_total_amount'] = np.log1p(df['total_amount'])

print('Số giá trị thiếu sau xử lý:')
display(df.isna().sum().to_frame('missing_count').T)

In [ ]:
# Gắn cờ outlier theo IQR trên total_amount
Q1 = df['total_amount'].quantile(0.25)
Q3 = df['total_amount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['is_high_value_invoice'] = ((df['total_amount'] < lower_bound) | (df['total_amount'] > upper_bound)).astype(int)

print(f'Q1 = {Q1:,.0f}')
print(f'Q3 = {Q3:,.0f}')
print(f'IQR = {IQR:,.0f}')
print(f'Lower bound = {lower_bound:,.0f}')
print(f'Upper bound = {upper_bound:,.0f}')
print('Số hóa đơn bị gắn cờ outlier:', int(df['is_high_value_invoice'].sum()))

display(df.loc[df['is_high_value_invoice'].eq(1), ['invoice_unique_key', 'buyer_name', 'total_amount', 'service_category']].head(10))

In [ ]:
# Tạo vài biến bổ sung để thuận tiện cho mô hình hóa và trình bày
revenue_bins = [-1, 1_000_000, 10_000_000, 50_000_000, np.inf]
revenue_labels = ['<=1M', '1M-10M', '10M-50M', '>50M']
df['revenue_band'] = pd.cut(df['total_amount'], bins=revenue_bins, labels=revenue_labels)
df['avg_tax_rate_est'] = np.where(df['subtotal_before_tax'] > 0, df['total_tax'] / df['subtotal_before_tax'], np.nan)
df['avg_tax_rate_est'] = df['avg_tax_rate_est'].fillna(df['avg_tax_rate_est'].median())

display(df[['invoice_unique_key', 'total_amount', 'revenue_band', 'avg_tax_rate_est', 'is_high_value_invoice']].head())

In [ ]:
# Lưu dữ liệu ra file để các notebook sau dùng ngay
prepared_path = BASE_DIR / 'prepared_invoice_dataset.csv'
cleaned_path = BASE_DIR / 'prepared_invoice_dataset_cleaned.csv'

df.to_csv(prepared_path, index=False, encoding='utf-8-sig')
df.to_csv(cleaned_path, index=False, encoding='utf-8-sig')

print(f'Đã lưu: {prepared_path.name}')
print(f'Đã lưu: {cleaned_path.name}')